In [1]:
from dataclasses import dataclass 
from spyral_utils.nuclear.nuclear_map import NucleusData
import numpy as np 


@dataclass 
class selection_rules_reaction:
    """
    Selection rules for a nuclear reaction
    
    Attributes
    ----------
        
    """
    beam_spin: float
    beam_parity: int 
    
    target_spin: float
    target_parity: int 

    targetlike_spin: float
    targetlike_parity: int

    def conservation_rules(self, cutoff_l: int)->tuple[float]:
        """ 
        Gets total spin and parity from the incoming channel

        """

        total_spin = np.arange(abs(self.beam_spin-self.target_spin), self.beam_spin+self.target_spin,1).tolist()
        if len(total_spin)==0:
                total_spin  = [abs(self.beam_spin-self.target_spin)]
        angular_momentum = range(0,cutoff_l+1,1)
        total_parity = [self.beam_parity*self.target_parity*(-1)**l for l in angular_momentum]
        return [total_spin,total_parity]
    
    def allowed_states(self, cutoff_l: int, beamlike_spin: tuple[float], beamlike_parity: tuple[int])->bool:
        total_spin, total_parity = self.conservation_rules(cutoff_l)

        for i in range(len(beamlike_parity)):
            allowed_spin = np.arange(abs(beamlike_spin[i]-self.targetlike_spin), beamlike_spin[i]+self.targetlike_spin,1).tolist()
            if len(allowed_spin)==0:
                allowed_spin = [abs(beamlike_spin[i]-self.targetlike_spin)]
            angular_momentum = range(0,cutoff_l+1,1)
            allowed_parity = [beamlike_parity[i]*self.targetlike_parity*(-1)**l for l in angular_momentum]

            # Find allowed states 
            idx_total_spin = []
            idx_allowed_spin = []
            for j in range(len(total_spin)):
                for k in range(len(allowed_spin)): 
                      if total_spin[j] == allowed_spin[k]: 
                           idx_total_spin.append(j)
                           idx_allowed_spin.append(k)
                      
            # Find which angular momentum it needs 
            for j in range(len(idx_total_spin)):
                parity_needed = total_parity[idx_total_spin[j]]
                l_values = [l for l in allowed_parity if l==parity_needed]


            print(total_spin, total_parity)
            print(allowed_spin,allowed_parity)
            print(idx_allowed_spin)
            print(allowed_spin[idx_allowed_spin], l_values)
        return True



In [60]:
group = selection_rules_reaction(
    beam_spin=3/2, 
    beam_parity=-1,

    target_spin=1,
    target_parity=+1,
    
    targetlike_spin=1/2,
    targetlike_parity=+1
)

In [61]:
group.allowed_states(cutoff_l=10, beamlike_spin=[4.], beamlike_parity=[+1])

[0.5, 1.5] [-1, 1, -1, 1, -1, 1, -1, 1, -1, 1, -1]
[3.5] [1, -1, 1, -1, 1, -1, 1, -1, 1, -1, 1]
[]


TypeError: list indices must be integers or slices, not list

In [40]:
import vector 
from spyral_utils.nuclear.nuclear_map import NuclearDataMap

nuclear_map = NuclearDataMap()

target = nuclear_map.get_data(1,2)
projectile = nuclear_map.get_data(4,7)
beam_energy = 98

target_vector = vector.array({
    "px": [0.0],
    "py": [0.0],
    "pz": [0.0],
    "E": [target.mass]
})

projectile_vector = vector.array({
    "px": [0.0],
    "py": [0.0],
    "pz": [np.sqrt(beam_energy*beam_energy + 2*projectile.mass)],
    "E": [projectile.mass + beam_energy]
})

entrance_energy = (target.mass + projectile.mass)
Qvalue = entrance_energy - nuclear_map.get_data(5,9).mass 

parent_vector = target_vector + projectile_vector
parent_vector_cm = parent_vector.boostCM_of(parent_vector)

Ecm = beam_energy*target.mass / (target.mass + projectile.mass) 
print("Qvalue", Qvalue)
print("Excitation Energy:", Ecm + Qvalue)

Qvalue 16.488234466036374
Excitation Energy: 38.34489458904227


In [65]:

target = nuclear_map.get_data(1,2)
projectile = nuclear_map.get_data(4,7)
ejectile = nuclear_map.get_data(0,1)
residual = nuclear_map.get_data(5,8)
beam_energy = 98

target_vector = vector.array({
    "px": [0.0],
    "py": [0.0],
    "pz": [0.0],
    "E": [target.mass]
})

projectile_vector = vector.array({
    "px": [0.0],
    "py": [0.0],
    "pz": [np.sqrt(beam_energy*beam_energy + 2*projectile.mass)],
    "E": [projectile.mass + beam_energy]
})


entrance_energy = (target.mass + projectile.mass)
exit_channel = (ejectile.mass + residual.mass)
Qvalue = entrance_energy  - exit_channel

parent_vector = target_vector + projectile_vector
parent_vector_cm = parent_vector.boostCM_of(parent_vector)

Ecm = beam_energy*target.mass / (target.mass + projectile.mass) 

# Approx relative energy 
E_d = nuclear_map.get_data(2,3).mass / projectile.mass *beam_energy
Erel = E_d*(nuclear_map.get_data(3,5).mass) / (nuclear_map.get_data(2,3).mass +nuclear_map.get_data(3,5).mass )

print("Qvalue", Qvalue)
print("Excitation Energy:", Erel+Qvalue)

Qvalue -2.0881667127432593
Excitation Energy: 24.20953486214384


In [66]:
Erel

26.2977015748871